In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from urllib.parse import unquote

In [ ]:
acm_tax = []
with open("../taxonomy/acm_ccs_2012.txt", 'r') as file:
    for line in file:
        line = line.strip().lower()
        line = re.sub(r'(_)', ' ', line)
        line = re.sub(r' / ', ' ', line)
        line = re.sub(r'/', ' ', line)
        acm_tax.append(line)
print(acm_tax)

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
acm_tax_single_word = {}
for tax in acm_tax:
    for word in tax.split(" "):
        if word in stop_words:
            continue
        word = re.sub(r'[^a-zA-Z\s-]', '', word)
        if word == "":
            continue
        acm_tax_single_word[word] = acm_tax_single_word.get(word, 0) + 1

print(acm_tax_single_word)

In [ ]:
onto = pd.read_csv("../taxonomy/CSO.3.3.csv", header=0, names=["c1", "c2", "c3"])
onto_cs = "<https://cso.kmi.open.ac.uk/topics/computer_science>"
onto_super_topic_of = "<http://cso.kmi.open.ac.uk/schema/cso#superTopicOf>"
cs_sub_topics = onto[(onto.c1 == onto_cs) & (onto.c2 == onto_super_topic_of)].c3.to_list()

In [ ]:
def get_topic_name(topic):
    t = re.sub(r'_', ' ', topic.split("/")[-1][:-1])
    t = unquote(t)
    return t

def clean_plurals_and_abbrv(all_topics):
    all_topics = sorted(set(all_topics))
    clean_topics = []
    for t in all_topics:
        abbrv = re.search(r'\(..+\)$', t)
        if abbrv is not None:
            non_abbrv = t[:abbrv.start()].strip()
            non_abbrv_initials = "".join(p[0] for p in non_abbrv.split(" "))
            abbrv = abbrv.group()[1:-1]
            if non_abbrv_initials == abbrv:
                clean_topics.append(non_abbrv)
                clean_topics.append(abbrv)
                continue
        clean_topics.append(t)
    all_topics = sorted(set(clean_topics))
    clean_topics = []
    topic_dict = {}
    for i in range(1, len(all_topics)):
        t = all_topics[i]
        if t.endswith("s") and topic_dict.get(t[:-1], None) is not None:
            continue
        clean_topics.append(t)
        topic_dict[t] = True
    return sorted(set(clean_topics))


all_topics = []
for topic in cs_sub_topics:
    subtopics = onto[(onto.c1 == topic) & (onto.c2 == onto_super_topic_of)].c3.to_list()
    subtopics = [get_topic_name(t) for t in subtopics]
    all_topics.append(get_topic_name(topic))
    all_topics.extend(subtopics)

clean_topics = clean_plurals_and_abbrv(all_topics)
print(len(clean_topics))
print(clean_topics)

In [ ]:
all_cs_tax = clean_plurals_and_abbrv(acm_tax + clean_topics)
custom_stop_words = ["accessibility", "education", "email", "publishing", "ranking", "sustainability"]
with open("../taxonomy/all_cs_taxonomy.txt", "w") as file:
    for t in all_cs_tax[:-1]:
        if t in custom_stop_words:
            continue
        file.write(t + "\n")
    if len(all_cs_tax) > 1:
        file.write(all_cs_tax[-1])
print(all_cs_tax)